# 08 Feature Engineering

## Objective

Objective:
Prepare model-ready features from the cleaned churn dataset using the findings from the earlier exploratory notebooks.

This step helps in:
- Converting cleaned data into modeling-friendly inputs
- Preserving strong churn signals discovered in bivariate and multivariate analysis
- Creating useful transformed and interaction features
- Preparing a consistent dataset for downstream feature selection and modeling


## Imports


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import seaborn as sns

project_root = Path.cwd()
if not (project_root / "src").exists() and (project_root.parent / "src").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.config import CATEGORICAL_COLUMNS, NUMERICAL_COLUMNS, TARGET_COLUMN
from src.data.data_loader import load_cleaned_data

sns.set_theme(style="whitegrid")


## Load Clean Data

Load the cleaned churn dataset and separate the target, numerical, and categorical feature groups that will be used in feature engineering.


In [2]:
df = load_cleaned_data()

target_column = TARGET_COLUMN
numerical_columns = [column for column in NUMERICAL_COLUMNS if column in df.columns]
categorical_columns = [column for column in CATEGORICAL_COLUMNS if column in df.columns]

feature_groups = {
    "target_column": target_column,
    "numerical_columns": numerical_columns,
    "categorical_columns": categorical_columns,
    "shape": df.shape,
}

feature_groups


{'target_column': 'Churn',
 'numerical_columns': ['tenure', 'MonthlyCharges', 'TotalCharges'],
 'categorical_columns': ['gender',
  'SeniorCitizen',
  'Partner',
  'Dependents',
  'PhoneService',
  'MultipleLines',
  'InternetService',
  'OnlineSecurity',
  'OnlineBackup',
  'DeviceProtection',
  'TechSupport',
  'StreamingTV',
  'StreamingMovies',
  'Contract',
  'PaperlessBilling',
  'PaymentMethod'],
 'shape': (7043, 21)}

## Revisit EDA Findings

Before engineering new features, summarize the main findings from notebooks `05`, `06`, and `07` so the next steps stay tied to the actual churn patterns already observed.

- From `05_univariate_analysis.ipynb`, the dataset showed uneven category sizes, meaningful structural labels such as `No internet service`, and numerical distributions that may need careful transformation instead of naive scaling.
- From `06_bivariate_analysis.ipynb`, the strongest single-feature churn signals were `tenure`, `MonthlyCharges`, `TotalCharges`, `Contract`, `PaymentMethod`, `TechSupport`, `OnlineSecurity`, and `InternetService`, while weaker features such as `gender` and `PhoneService` showed much less separation.
- From `07_multivariate_analysis.ipynb`, `tenure` and `TotalCharges` were strongly related, and high-risk combinations such as `Month-to-month + Electronic check` and `Fiber optic + No TechSupport` suggested that interaction features may add value during modeling.

These findings guide the feature-engineering stage: preserve strong churn signals, handle redundant lifecycle features carefully, retain meaningful structural categories, and consider interaction features where the combined risk pattern is sharper than the single-feature pattern.


## Handle Categorical Encoding

Encode categorical features according to their business meaning.

- Binary features are mapped to `0/1`.
- `Contract` is ordinal-encoded because it reflects increasing customer commitment from `Month-to-month` to `Two year`.
- Multi-class service and payment features are one-hot encoded so structural labels such as `No internet service` remain distinct from ordinary `No` values.


In [3]:
encoded_df = df.copy()

binary_mappings = {
    "gender": {"Female": 0, "Male": 1},
    "Partner": {"No": 0, "Yes": 1},
    "Dependents": {"No": 0, "Yes": 1},
    "PhoneService": {"No": 0, "Yes": 1},
    "PaperlessBilling": {"No": 0, "Yes": 1},
    target_column: {"No": 0, "Yes": 1},
}

for column, mapping in binary_mappings.items():
    if column in encoded_df.columns:
        encoded_df[column] = encoded_df[column].map(mapping)

if "SeniorCitizen" in encoded_df.columns:
    encoded_df["SeniorCitizen"] = encoded_df["SeniorCitizen"].astype(int)

contract_mapping = {
    "Month-to-month": 0,
    "One year": 1,
    "Two year": 2,
}
if "Contract" in encoded_df.columns:
    encoded_df["Contract"] = encoded_df["Contract"].map(contract_mapping)

one_hot_columns = [
    column
    for column in [
        "MultipleLines",
        "InternetService",
        "OnlineSecurity",
        "OnlineBackup",
        "DeviceProtection",
        "TechSupport",
        "StreamingTV",
        "StreamingMovies",
        "PaymentMethod",
    ]
    if column in encoded_df.columns
]

encoded_df = pd.get_dummies(
    encoded_df,
    columns=one_hot_columns,
    drop_first=False,
    dtype=int,
)

encoding_summary = {
    "binary_encoded": [column for column in binary_mappings if column in df.columns],
    "ordinal_encoded": [column for column in ["Contract"] if column in df.columns],
    "one_hot_encoded": one_hot_columns,
    "encoded_shape": encoded_df.shape,
}

encoding_summary


{'binary_encoded': ['gender',
  'Partner',
  'Dependents',
  'PhoneService',
  'PaperlessBilling',
  'Churn'],
 'ordinal_encoded': ['Contract'],
 'one_hot_encoded': ['MultipleLines',
  'InternetService',
  'OnlineSecurity',
  'OnlineBackup',
  'DeviceProtection',
  'TechSupport',
  'StreamingTV',
  'StreamingMovies',
  'PaymentMethod'],
 'encoded_shape': (7043, 40)}

In [4]:
encoded_df.head()


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,Contract,PaperlessBilling,MonthlyCharges,...,StreamingTV_No,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No,StreamingMovies_No internet service,StreamingMovies_Yes,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,7590-VHVEG,0,0,1,0,1,0,0,1,29.85,...,1,0,0,1,0,0,0,0,1,0
1,5575-GNVDE,1,0,0,0,34,1,1,0,56.95,...,1,0,0,1,0,0,0,0,0,1
2,3668-QPYBK,1,0,0,0,2,1,0,1,53.85,...,1,0,0,1,0,0,0,0,0,1
3,7795-CFOCW,1,0,0,0,45,0,1,0,42.30,...,1,0,0,1,0,0,1,0,0,0
4,9237-HQITU,0,0,0,0,2,1,0,1,70.70,...,1,0,0,1,0,0,0,0,1,0


## Numerical Feature Transformations

Apply numerical transformations where they support modeling.

- Keep original numerical columns intact.
- Create a new log-transformed column for `TotalCharges` because it is the main skewed numerical feature.
- Add new standardized columns such as `tenure_scaled` and `MonthlyCharges_scaled` for scale-sensitive models, while preserving the original values for tree-based models.


In [8]:
transformed_df = encoded_df.copy()

if "TotalCharges" in transformed_df.columns:
    transformed_df["TotalCharges_log"] = np.log1p(transformed_df["TotalCharges"])

scaling_columns = [
    column for column in ["tenure", "MonthlyCharges", "TotalCharges"]
    if column in transformed_df.columns
]

scaled_column_names = []
for column in scaling_columns:
    column_mean = transformed_df[column].mean()
    column_std = transformed_df[column].std(ddof=0)
    scaled_column_name = f"{column}_scaled"
    if column_std:
        transformed_df[scaled_column_name] = (
            transformed_df[column] - column_mean
        ) / column_std
    else:
        transformed_df[scaled_column_name] = 0.0
    scaled_column_names.append(scaled_column_name)

scaled_numerical_df = transformed_df[scaled_column_names].copy()

transformation_summary = {
    "log_transformed": [column for column in ["TotalCharges_log"] if column in transformed_df.columns],
    "scaled_columns": scaled_column_names,
    "transformed_shape": transformed_df.shape,
    "scaled_numerical_shape": scaled_numerical_df.shape,
}

transformation_summary


{'log_transformed': ['TotalCharges_log'],
 'scaled_columns': ['tenure_scaled',
  'MonthlyCharges_scaled',
  'TotalCharges_scaled'],
 'transformed_shape': (7043, 44),
 'scaled_numerical_shape': (7043, 3)}

In [9]:
transformed_df[[column for column in ["TotalCharges", "TotalCharges_log"] if column in transformed_df.columns]].head()


,TotalCharges,TotalCharges_log
0,29.85,3.429137
1,1889.50,7.544597
2,108.15,4.692723
3,1840.75,7.518471
4,151.65,5.028148


In [10]:
transformed_df[[column for column in scaling_columns + scaled_column_names if column in transformed_df.columns]].head()


,tenure,MonthlyCharges,TotalCharges,tenure_scaled,MonthlyCharges_scaled,TotalCharges_scaled
0,1,29.85,29.85,-1.277445,-1.160323,-0.992611
1,34,56.95,1889.50,0.066327,-0.259629,-0.172165
2,2,53.85,108.15,-1.236724,-0.362660,-0.958066
3,45,42.30,1840.75,0.514251,-0.746535,-0.193672
4,2,70.70,151.65,-1.236724,0.197365,-0.938874


## Create Interaction Features

Create interaction features where churn risk became sharper when variables were combined in multivariate analysis.

- `tenure x MonthlyCharges` captures the joint effect of customer age and monthly bill level.
- `Contract x PaymentMethod` captures risk patterns such as `Month-to-month + Electronic check`.
- `InternetService x TechSupport` and `InternetService x OnlineSecurity` preserve the high-risk service combinations observed in the interaction heatmaps.


In [11]:
interaction_df = transformed_df.copy()

if {"tenure", "MonthlyCharges"}.issubset(interaction_df.columns):
    interaction_df["tenure_x_MonthlyCharges"] = (
        interaction_df["tenure"] * interaction_df["MonthlyCharges"]
    )

payment_method_columns = [
    column for column in interaction_df.columns
    if column.startswith("PaymentMethod_")
]
for column in payment_method_columns:
    interaction_df[f"Contract_x_{column}"] = interaction_df["Contract"] * interaction_df[column]

internet_service_columns = [
    column for column in interaction_df.columns
    if column.startswith("InternetService_")
]
tech_support_columns = [
    column for column in interaction_df.columns
    if column.startswith("TechSupport_")
]
online_security_columns = [
    column for column in interaction_df.columns
    if column.startswith("OnlineSecurity_")
]

for internet_column in internet_service_columns:
    for support_column in tech_support_columns:
        interaction_df[f"{internet_column}_x_{support_column}"] = (
            interaction_df[internet_column] * interaction_df[support_column]
        )

for internet_column in internet_service_columns:
    for security_column in online_security_columns:
        interaction_df[f"{internet_column}_x_{security_column}"] = (
            interaction_df[internet_column] * interaction_df[security_column]
        )

interaction_columns = [
    column for column in interaction_df.columns
    if "_x_" in column
]

interaction_summary = {
    "interaction_count": len(interaction_columns),
    "sample_interactions": interaction_columns[:10],
    "interaction_shape": interaction_df.shape,
}

interaction_summary


{'interaction_count': 23,
 'sample_interactions': ['tenure_x_MonthlyCharges',
  'Contract_x_PaymentMethod_Bank transfer (automatic)',
  'Contract_x_PaymentMethod_Credit card (automatic)',
  'Contract_x_PaymentMethod_Electronic check',
  'Contract_x_PaymentMethod_Mailed check',
  'InternetService_DSL_x_TechSupport_No',
  'InternetService_DSL_x_TechSupport_No internet service',
  'InternetService_DSL_x_TechSupport_Yes',
  'InternetService_Fiber optic_x_TechSupport_No',
  'InternetService_Fiber optic_x_TechSupport_No internet service'],
 'interaction_shape': (7043, 67)}

In [12]:
interaction_preview_columns = [
    column for column in [
        "tenure",
        "MonthlyCharges",
        "tenure_x_MonthlyCharges",
        "Contract",
        "PaymentMethod_Electronic check",
        "Contract_x_PaymentMethod_Electronic check",
        "InternetService_Fiber optic",
        "TechSupport_No",
        "InternetService_Fiber optic_x_TechSupport_No",
        "OnlineSecurity_No",
        "InternetService_Fiber optic_x_OnlineSecurity_No",
    ]
    if column in interaction_df.columns
]

interaction_df[interaction_preview_columns].head()


,tenure,MonthlyCharges,tenure_x_MonthlyCharges,Contract,PaymentMethod_Electronic check,Contract_x_PaymentMethod_Electronic check,InternetService_Fiber optic,TechSupport_No,InternetService_Fiber optic_x_TechSupport_No,OnlineSecurity_No,InternetService_Fiber optic_x_OnlineSecurity_No
0,1,29.85,29.85,0,1,0,0,1,0,1,0
1,34,56.95,1936.30,1,0,0,0,1,0,0,0
2,2,53.85,107.70,0,0,0,0,1,0,0,0
3,45,42.30,1903.50,1,0,0,0,0,0,0,0
4,2,70.70,141.40,0,1,0,1,1,1,1,1
